## Introducción

La tesoreria general de la republica entrega los datos que consolidan las patentes por no uso de derechos de aprovechamiento de aguas en Chile.
Este conjunto de datos permite clasificar las tesorerías regionales con más patentes por no uso de aprovechamiento de aguas además de permitir clasificar si los ddeudores son 
personas naturales o personas Juridicas.

**Objetivo general (Fase 2).** Construir un *pipeline* reproducible que deje el *dataset*
limpio, codificado y escalado, listo para el modelado posterior.

**Objetivos específicos.**
- Obtener y explorar los datos verificando su estructura y calidad.
- Limpiar gestionando rigurosamente los valores nulos.
- Transformar las variables categóricas nominales con codificación One-Hot.
- Estandarizar las variables continuas.
- Validar técnicamente el resultado.

**Atributos:** `id`, `gender`, `age`, `hypertension`, `heart_disease`, `ever_married`,
`work_type`, `Residence_type`, `avg_glucose_level`, `bmi`, `smoking_status` y la variable
objetivo `stroke` (1 = sufrió accidente cerebrovascular, 0 = no).

### Librerías utilizadas

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Cálculo numérico y manejo de arreglos. |
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `matplotlib` | Generar los gráficos (*boxplots*). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar
dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

In [ ]:
import sys                                   # intérprete del kernel
import subprocess                            # instalar paquetes faltantes
import importlib                             # refrescar el sistema de importación
import importlib.util                        # comprobar si un paquete está instalado

# Verificación del entorno: instala en el intérprete del kernel lo que falte.
REQUERIDOS = {                               # módulo: nombre del paquete en pip
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",               # se importa como sklearn, se instala como scikit-learn
}
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                           # cálculo numérico y operaciones vectorizadas
import pandas as pd                          # estructuras tabulares: Series y DataFrame
import matplotlib.pyplot as plt              # gráficos de control
from sklearn import preprocessing            # LabelEncoder y OneHotEncoder
from sklearn.preprocessing import StandardScaler   # estandarización z-score

# Reproducibilidad: fijamos la semilla aleatoria del entorno
np.random.seed(42)

%matplotlib inline

In [ ]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("numpy", np), ("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno, no en la
    carpeta desde la que se lanzó JupyterLab. Como cada cuaderno vive en la carpeta
    de su fase (F1, F2, F3 o F4), si la carpeta de trabajo es una de ellas la raíz
    es la carpeta superior. Si el cuaderno se ejecuta desde la raíz, se usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()                 # la misma función del cuaderno de la Fase 1
DIR_CRUDO = RAIZ / "data" / "raw"            # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # resultado del pipeline
DIR_DOCS = RAIZ / "docs"                     # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "tb_patentes_aguas_december_2025_data.csv"
print("\nArchivo esperado en:", ARCHIVO)

### Procedencia del conjunto de datos

| Campo | Valor |
| --- | --- |
| Título | consolidacion de patentes por no uso de derechos de aprovechamiento de aguas |
| Autor | Tesoreía General de La República |
| Plataforma | Datos.gob.cl |
| Enlace | `https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715` |
| Estructura esperada | 46.468 filas × 21 columnas originales· 34 nulos en `tesoreria_regional` |
| Unidad de observación | Pago patentes de agua|

**Referencia en APA 7 para el informe:**

> **Referencia en APA 7 para el informe:**
Tesorería General de la República. (2025). Reporte consolidado de recaudación y cartera morosa de patentes de aguas (diciembre 2025) [Conjunto de datos]. Ministerio de Hacienda, Gobierno de Chile. https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715

Documentar la procedencia no es un trámite: sin ella, nadie puede verificar sobre
qué versión del archivo se trabajó.

## 1. Obtención de los datos

In [ ]:
def cargar_datos(ruta):
    """
    Carga un conjunto de datos desde un archivo CSV.

    Parámetros
    ----------
    ruta : str o Path
        Ruta del archivo CSV.

    Retorna
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Lanza
    -----
    FileNotFoundError
        Si el archivo no existe, con un mensaje que indica dónde debe estar.
    """
    ruta = Path(ruta)
    try:
        # En este archivo los faltantes de bmi vienen como el texto 'N/A':
        # na_values los convierte en NaN al leer.
        df = pd.read_csv(ruta, na_values=["N/A"])
    except FileNotFoundError:
        # from None oculta la traza interna de pandas: queda solo el mensaje útil.
        raise FileNotFoundError(
            f"No se encontró {ruta.name} en {ruta.parent.resolve()}.\n"
            "Descárguelo desde el enlace de la tabla de procedencia y guárdelo en "
            "data/raw/ de la raíz del proyecto."
        ) from None
    print(f"Datos cargados: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df


df_crudo = cargar_datos(ARCHIVO)

# Se conserva el original sin modificaciones y se trabaja sobre una copia
df = df_crudo.copy()
df.head()


 **Se agrega esta celda para eliminar RUT_ROL y RUT_ROL_DV** Agreagr aca o no???. O cambiar RUT_ROL por un identificador simple, y dijito verificador eliminar la columna.

## 2. Exploración inicial

**Qué hace este paso.** Antes de modificar nada, miramos el estado original de los datos.
Esto nos da la "línea base" contra la cual verificaremos después cada transformación.

In [ ]:
def explorar_dataframe(df):
    """Resumen exploratorio: dimensiones, tipos, nulos y estadisticos descriptivos."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos (variables numericas):")
    return df.describe()


explorar_dataframe(df)

In [ ]:
# Conteo de categorias en las variables nominales (control de calidad de entrada)
columnas_categoricas = ['RUT_ROL', 'TIPO_PERSONA', 'RUT_ROL_DV',
                        'TESORERIA', 'TESORERIA_REGIONAL', 'COMUNA',
                        'PROVINCIA', 'OFICINA_CBR', 'OFICINA',
                        'CONCESION', 'MOV_TIPO']
for col in columnas_categoricas:
    print(f"\n{col}:")
    # dropna=False incluye los nulos como una categoría más: si una variable
    # tiene faltantes queremos verlos aquí, no que desaparezcan del conteo.
    print(df[col].value_counts(dropna=False))

### El diccionario de variables

Antes de limpiar hay que declarar **qué es cada variable**. Esta es la decisión
que determina todo el preprocesamiento posterior.

La idea central: **el tipo de dato no dice qué es una variable**. En este
conjunto `hypertension`, `id` y `age` son todas numéricas, y sin embargo exigen
tratamientos completamente distintos.

| Rol analítico | Qué preprocesamiento exige |
| --- | --- |
| **Continua** | Detección de atípicos, imputación, escalamiento |
| **Binaria** | Codificación 0/1 y revisión de desbalance |
| **Nominal** | *One-hot encoding* |
| **Ordinal** | Codificación con el orden declarado explícitamente |
| **Identificador** | Verificar unicidad y excluir del análisis |
| **Objetivo** | Se preserva sin transformar en esta fase |

In [ ]:
# El diccionario se DECLARA con lo que se sabe del dominio y se COMPLETA con lo
# observado en el archivo: no se afirma nada que no se verifique.
DECLARADO = [
    ("CUENTA", "identificador", "Identificador único de la cuenta"),
    ("RUT_ROL", "identificador", "RUT del titular, sin dígito verificador (dato personal: no se publica)"),
    ("TIPO_PERSONA", "binaria", "Tipo de persona registrado"),
    ("RUT_ROL_DV", "nominal", "Dígito verificador del RUT"),
    ("FORM_FOLIO", "continua", "Número de folio del formulario"),
    ("MONTO", "continua", "Monto asociado a la patente"),
    ("SALDO", "continua", "Saldo registrado de la patente"),
    ("PAGADO", "continua", "Monto pagado"),
    ("FECHA", "fecha", "Fecha del registro"),
    ("MOV_TIPO", "discreta", "Código del tipo de movimiento"),
    ("TESORERIA", "alta cardinalidad", "Tesorería asociada al registro"),
    ("TESORERIA_REGIONAL", "alta cardinalidad", "Tesorería regional correspondiente"),
    ("COMUNA", "alta cardinalidad", "Comuna asociada a la patente"),
    ("CONCESION", "alta cardinalidad", "Identificación de la concesión"),
    ("PROVINCIA", "alta cardinalidad", "Provincia asociada al registro"),
    ("OFICINA_CBR", "alta cardinalidad", "Oficina del Conservador de Bienes Raíces"),
    ("N_DE_CORRELATIVO_O_PATENTE", "continua", "Número correlativo o de patente"),
    ("ANIO_PATENTE", "discreta", "Año correspondiente a la patente"),
    ("UTM", "continua", "Valor registrado en UTM"),
    ("OFICINA", "alta cardinalidad", "Oficina asociada al registro"),
    ("FECHA_EJECUCION", "fecha", "Fecha de ejecución del proceso"),
]
diccionario = pd.DataFrame(DECLARADO, columns=["variable", "rol_analitico", "descripcion"])

# Columnas OBSERVADAS: se leen del archivo, no se escriben a mano
diccionario["dtype"] = [str(df_crudo[v].dtype) for v in diccionario["variable"]]
# nunique() cuenta valores distintos: distingue una binaria de una continua
diccionario["n_unicos"] = [int(df_crudo[v].nunique()) for v in diccionario["variable"]]
# Sobre booleanos, mean() devuelve la PROPORCIÓN de True: por 100 da el porcentaje
diccionario["pct_nulos"] = [round(df_crudo[v].isna().mean() * 100, 1)
                            for v in diccionario["variable"]]

# El diccionario es un entregable, no una tabla de paso: se guarda. Reemplaza al de la
# Fase 1 a propósito: conserva sus columnas y le agrega lo observado en el archivo.
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
diccionario

### Valores atípicos: medir antes de decidir

La sección siguiente imputa con la mediana «porque hay valores extremos». Esa
afirmación hay que **demostrarla**, no enunciarla. Se usa el criterio del rango
intercuartílico:

$$\text{atípico si} \quad x < Q_1 - 1{,}5 \cdot \text{RIC} \quad \text{o} \quad x > Q_3 + 1{,}5 \cdot \text{RIC}$$

In [ ]:
def detectar_atipicos_iqr(serie, factor=1.5):
    """Identifica valores atípicos por el criterio del rango intercuartílico.

    Retorna
    -------
    tuple(np.ndarray, float, float)
        Máscara booleana de atípicos, límite inferior y límite superior.
    """
    # dropna() es obligatorio: un NaN propagaría y devolvería NaN como cuartil
    valores = serie.dropna().to_numpy(dtype="float64")

    # np.percentile con una lista devuelve varios percentiles de una vez
    q1, q3 = np.percentile(valores, [25, 75])
    ric = q3 - q1                                  # rango intercuartílico
    limite_inf, limite_sup = q1 - factor * ric, q3 + factor * ric

    # El operador | es el "o" elemento a elemento de pandas. Con 'or' fallaría:
    # 'or' espera un único booleano, no una serie completa.
    mascara = (serie < limite_inf) | (serie > limite_sup)
    # Comparar con NaN da False pero deja NA: fillna(False) lo hace explícito
    return mascara.fillna(False).to_numpy(), limite_inf, limite_sup


filas = []
for col in ["MONTO", "SALDO", "PAGADO","UTM"]:
    mascara, inf, sup = detectar_atipicos_iqr(df[col])
    filas.append({
        "variable": col,
        "limite_inf": round(inf, 2),
        "limite_sup": round(sup, 2),
        "n_atipicos": int(mascara.sum()),          # sobre booleanos, sum() cuenta True
        "pct_atipicos": round(mascara.mean() * 100, 2),
        "media": round(df[col].mean(), 2),
        "mediana": round(df[col].median(), 2),
    })
pd.DataFrame(filas)

## 3. Limpieza de datos

Limpiar significa dejar los datos completos y consistentes. Se toman las siguientes
decisiones, cada una justificada con evidencia del propio dataset (46.467 filas).

**a) Eliminar `RUT_ROL`.** Es un identificador del titular o del rol, no único por fila:
presenta 4.504 valores distintos, por lo que un mismo valor se repite en varias patentes.
No contiene información que ayude a predecir si existe o no deuda, ya que, por su alta
cardinalidad, un modelo solo memorizaría casos particulares sin poder generalizar a
titulares nuevos.

**b) Eliminar `RUT_ROL_DV`.** Presenta una única categoría (un espacio en blanco) en las
46.467 filas y no tiene nulos. Al no variar entre patentes, no aporta información.

**c) Eliminar `CUENTA`.** Es un identificador único por fila (46.467 valores distintos en
46.467 filas). Es un código que individualiza cada registro y no una característica de la
patente, por lo que no ayuda a predecir si existe o no deuda.

**d) Eliminar `FORM_FOLIO`.** Es un código del formulario de emisión, no una característica
de la patente ni una cantidad con sentido numérico. Se repite entre filas (6.787 valores
distintos) y mezcla formatos distintos (valores del orden de miles y 690 del orden de 10⁹).

**e) Eliminar `MOV_TIPO`.** Tiene un único valor (1) en todas las filas.

**f) Eliminar `FECHA_EJECUCION`.** Tiene un único valor (2025-12-01), que corresponde solo a
la fecha de extracción del dato.

**g) Eliminar `TESORERIA`.** Es redundante con `TESORERIA_REGIONAL`: cada tesorería
pertenece a una única región, y tiene más categorías (46 contra 18) sin aportar
información nueva.

**h) Eliminar `OFICINA`.** Tiene 303 categorías, por lo que su codificación generaría
cientos de columnas con pocas observaciones. Además coincide con `COMUNA` en el 62 % de las
filas.

**i) Eliminar `OFICINA_CBR`.** Tiene 137 categorías y el valor más frecuente es
"SIN CLASIFICACION" (10.315 filas, 22 %), lo que revela baja calidad del dato.

**j) Eliminar `COMUNA`.** Tiene 319 categorías (alta cardinalidad). La información
territorial se conserva en `TESORERIA_REGIONAL` (18 categorías), que es el nivel de análisis
del proyecto. Se declara como limitación la pérdida de detalle comunal.

**k) Eliminar `PROVINCIA`.** Tiene 56 categorías y 691 nulos (1,5 %). Se conserva la
información territorial en `TESORERIA_REGIONAL`.

**l) Eliminar `CONCESION`.** Es un código que vale 0 en 45.961 de las 46.467 filas
(98,9 %). Los 506 registros restantes presentan 384 valores distintos, casi todos con una o
dos apariciones, por lo que la variable no permite identificar patrones generalizables.

**m) Eliminar `N_DE_CORRELATIVO_O_PATENTE`.** Es un número correlativo de la patente (valores de 1 a 12.458), por lo que funciona como un código y no como una cantidad. Se repite entre filas (7.331 valores distintos en 45.776 registros con dato) y no muestra una relación ordenada con la morosidad. Su única señal aparente es temporal (correlación de 0,36 con ANIO_PATENTE), información que ya se conserva en esa variable. Además presenta 691 nulos (1,5 %).

**Valores nulos.** Las variables continuas (`MONTO`, `SALDO`, `PAGADO`, `UTM`) no
contienen nulos, por lo que no es necesaria ninguna imputación con media, mediana u otro
estadístico. Sí hay nulos en `TESORERIA_REGIONAL` (34), `ANIO_PATENTE` (691). 

<!--**b) Imputar `bmi` con la mediana.** `bmi` (índice de masa corporal) es la única columna
con valores faltantes. "Imputar" es rellenar esos huecos con un valor representativo.
Tenemos dos opciones:
- La **media** (promedio): se ve arrastrada por los valores extremos.
- La **mediana** (valor central): es **robusta** ante valores atípicos.

Como el `bmi` tiene valores atípicos (se ven en el *boxplot* del final), usar la mediana
evita que esos extremos sesguen el relleno. La función `imputar_nulos_numericos` permite
elegir la estrategia con un parámetro e informa cuántos nulos rellenó y con qué valor
(trazabilidad).-->

In [ ]:
def imputar_nulos_numericos(df, columna, estrategia="mediana"):
    """
    Imputa los valores nulos de una columna numerica.

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna numerica a imputar.
    estrategia : str
        'media' o 'mediana'.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna imputada.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")
    # isnull() da True/False por celda; sum() cuenta los True.
    # Se calcula ANTES de imputar: después ya no habría nulos que contar.
    n_nulos = int(df[columna].isnull().sum())
    if estrategia == "media":
        valor = df[columna].mean()
    elif estrategia == "mediana":
        valor = df[columna].median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'")
    df = df.copy()
    df[columna] = df[columna].fillna(valor)
    print(f"'{columna}': {n_nulos} nulos imputados con la {estrategia} = {valor:.2f}")
    return df

In [ ]:
df.isnull().sum()

In [ ]:
# Antes de descartarlo se comprueba que 'id' sea efectivamente un identificador:
# is_unique devuelve True si no hay valores repetidos.

# drop(columns=[...]) devuelve un DataFrame NUEVO sin esa columna.
df = df.drop(columns=['RUT_ROL','RUT_ROL_DV','CUENTA','FORM_FOLIO','MOV_TIPO','FECHA_EJECUCION','TESORERIA','OFICINA','OFICINA_CBR','COMUNA','PROVINCIA','CONCESION','N_DE_CORRELATIVO_O_PATENTE'])
df.head()

In [ ]:
df.isnull().sum()

**Tratamiento de valores nulos.** Las variables continuas (`MONTO`, `SALDO`, `PAGADO`,
`UTM`) no contienen nulos, por lo que no se aplica imputación con media, mediana u otro
estadístico. Los nulos se concentran en dos variables, y cada una recibe un tratamiento:

**`TESORERIA_REGIONAL` (34 nulos, 0,07 % de las filas): se eliminan las filas.**
La variable es nominal, por lo que la mediana no es aplicable, e imputarla con la moda
asignaría esas patentes a una región sin fundamento. Tampoco puede recuperarse desde
`COMUNA`, porque una misma comuna se distribuye entre varias tesorerías regionales. Como
su peso es mínimo (0,05 % del saldo impago total), eliminarlas no altera de forma relevante
el análisis. Como limitación, 33 de las 34 filas presentan saldo impago, una proporción
mayor al promedio, pero con un efecto despreciable por su cantidad.

**`ANIO_PATENTE` (691 nulos, 1,5 % de las filas): se rellena con el valor centinela `0`.**
Se prefirió no eliminar estas filas porque concentran el 1,4 % del saldo impago total, que
sigue siendo válido para el análisis por Tesorería Regional y tipo de titular, y porque
corresponden casi en su totalidad (690 de 691) a registros de 2025, con una tasa de
morosidad distinta al resto. Tampoco se imputa con la mediana, porque inventaría un año
para patentes cuyo año se desconoce. El valor `0` no es un año real: la variable se
mantiene numérica, y se agrega la columna indicadora `anio_faltante` (1 = sin dato) para
identificar siempre esas filas. Como consecuencia, al calcular estadísticos o tendencias
por año debe excluirse el valor `0`.

**Resultado:** el conjunto queda con 46.433 filas y sin valores nulos.


In [ ]:
# --- Tratamiento de valores nulos (decisión de limpieza F2) ---
# Se parte de df (ya sin las columnas descartadas) y se trabaja sobre una copia.
df_nulos = df.copy()
n_antes = len(df_nulos)

# 1) TESORERIA_REGIONAL: eliminar las filas con nulo
df_nulos = df_nulos.dropna(subset=['TESORERIA_REGIONAL']).copy()
print(f"Filas eliminadas: {n_antes - len(df_nulos)} "
      f"({(n_antes - len(df_nulos)) / n_antes * 100:.2f} %) | restantes: {len(df_nulos)}")

# 2) ANIO_PATENTE: marcar los faltantes y rellenar con el valor centinela 0
# El indicador se crea ANTES del relleno, para no perder qué filas no tenían dato.
df_nulos['anio_faltante'] = df_nulos['ANIO_PATENTE'].isna().astype(int)
df_nulos['ANIO_PATENTE'] = df_nulos['ANIO_PATENTE'].fillna(0).astype(int)

# 3) Validación con evidencia visible
print("\nNulos por columna tras el tratamiento:")
print(df_nulos.isna().sum())
print(f"\nFilas con ANIO_PATENTE = 0: {(df_nulos['ANIO_PATENTE'] == 0).sum()}")
print(f"Suma de anio_faltante: {df_nulos['anio_faltante'].sum()}")
print(f"Tipo de ANIO_PATENTE: {df_nulos['ANIO_PATENTE'].dtype}")

df = df_nulos

## 4. Transformación: codificación One-Hot

Los modelos solo entienden números, no texto. Por eso hay que convertir las variables de
texto en números. Pero **cómo** las convertimos importa:

- `TESORERIA_REGIONAL` y `TIPO_PERSONA` son
  **categóricas nominales**: sus categorías **no tienen un orden**.
- Si simplemente les pusiéramos 0, 1, 2..., el modelo creería que existe un orden falso
  (que "2" vale más que "1"). Para evitarlo usamos **One-Hot Encoding**: cada categoría se
  convierte en su propia columna de 0 y 1.

En este caso, **`LabelEncoder`** no aplica

Mantenemos el método visto en el curso —**`LabelEncoder` seguido de `OneHotEncoder`**— pero
lo encapsulamos en una sola función para no repetir el mismo bloque cinco veces.

### El patrón `fit` / `transform`

Casi todas las herramientas de scikit-learn funcionan en dos tiempos:
- **`fit`** = *aprender*: la herramienta mira los datos y guarda lo que necesita (por
  ejemplo, qué categorías existen).
- **`transform`** = *aplicar*: usa lo aprendido para convertir los datos.

Dentro de la función `codificar_one_hot` este patrón aparece **dos veces** (una con
`LabelEncoder` y otra con `OneHotEncoder`). Paso a paso, con el ejemplo `["Yes","No","Yes","No"]`:

1. **`le = preprocessing.LabelEncoder()`** — crea la herramienta que convierte texto en
   enteros. Aún está vacía.
2. **`le.fit(datos)`** — aprende las categorías y les asigna un número en orden alfabético:
   `"No" → 0`, `"Yes" → 1`.
3. **`le.transform(datos)`** — aplica lo aprendido → `[1, 0, 1, 0]`. Esto ya es *Label
   Encoding*.
4. **`ohe = preprocessing.OneHotEncoder()`** — crea la segunda herramienta, para One-Hot.
5. **`d = datos_codificados.reshape(-1, 1)`** — reorganiza los datos de una fila a una
   columna, porque el `OneHotEncoder` exige formato de tabla (2 dimensiones). El `-1`
   significa "calcula tú las filas" y el `1` significa "una columna".
6. **`ohe.fit(d)` y `ohe.transform(d).toarray()`** — expande cada entero en columnas
   binarias. `.toarray()` convierte el resultado a un arreglo normal.
7. **`pd.DataFrame(...)` + `pd.concat(...)`** — empaqueta las nuevas columnas con nombres
   legibles y las une al `DataFrame`, eliminando la columna original.

**Mejoras respecto al código original:** la función (a) asigna los nombres legibles
directamente, en lugar de renombrar por número de posición; (b) conserva el índice original
(`index=df.index`) para que la unión no desalinee filas; y (c) **valida** que la cantidad
de nombres coincida con la cantidad de categorías, avisando con un error claro si no.

In [ ]:
def codificar_one_hot(df, columna, nombres_columnas):
    """
    Codifica una variable categorica nominal aplicando LabelEncoder y luego
    OneHotEncoder (patron fit/transform de scikit-learn).

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna categorica a codificar.
    nombres_columnas : list[str]
        Nombres de las nuevas columnas binarias. El orden debe coincidir con el
        orden alfabetico de las categorias (criterio interno de LabelEncoder).

    Retorna
    -------
    pd.DataFrame
        DataFrame con las nuevas columnas one-hot y sin la columna original.

    Lanza
    -----
    KeyError
        Si la columna no existe.
    ValueError
        Si el numero de nombres no coincide con el numero de categorias.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")

    # Paso 1-3: LabelEncoder aprende las categorias y las convierte a enteros
    le = preprocessing.LabelEncoder()
    datos = df[columna]
    le.fit(datos)
    datos_codificados = le.transform(datos)

    # Verificacion: los nombres deben corresponder a las categorias detectadas
    if len(nombres_columnas) != len(le.classes_):
        raise ValueError(
            f"Se esperaban {len(le.classes_)} nombres para '{columna}' "
            f"({list(le.classes_)}), pero se recibieron {len(nombres_columnas)}."
        )

    # Paso 4-6: OneHotEncoder expande los enteros a columnas binarias
    ohe = preprocessing.OneHotEncoder()
    d = datos_codificados.reshape(-1, 1)   # formato 2D requerido por el encoder
    ohe.fit(d)
    matriz = ohe.transform(d).toarray()

    # Paso 7: empaquetamos con nombres legibles y conservamos el indice original
    nuevas = pd.DataFrame(matriz, columns=nombres_columnas, index=df.index).astype(int)

    df = df.drop(columns=[columna]).reset_index(drop=True)
    nuevas = nuevas.reset_index(drop=True)
    return pd.concat([df, nuevas], axis=1)

### a. `TESORERIA_REGIONAL`
| Columna generada | Categoría original de `TESORERIA_REGIONAL` |
|---|---|
| `teso_antofagasta` | TESORERÍA REGIONAL DE ANTOFAGASTA |
| `teso_arica_parinacota` | TESORERÍA REGIONAL DE ARICA Y PARINACOTA |
| `teso_atacama` | TESORERÍA REGIONAL DE ATACAMA |
| `teso_aysen` | TESORERÍA REGIONAL DE AYSEN |
| `teso_coquimbo` | TESORERÍA REGIONAL DE COQUIMBO |
| `teso_araucania` | TESORERÍA REGIONAL DE LA ARAUCANIA |
| `teso_los_lagos` | TESORERÍA REGIONAL DE LOS LAGOS |
| `teso_los_rios` | TESORERÍA REGIONAL DE LOS RIOS |
| `teso_magallanes` | TESORERÍA REGIONAL DE MAGALLANES |
| `teso_tarapaca` | TESORERÍA REGIONAL DE TARAPACA |
| `teso_valparaiso` | TESORERÍA REGIONAL DE VALPARAÍSO |
| `teso_nuble` | TESORERÍA REGIONAL DE ÑUBLE |
| `teso_biobio` | TESORERÍA REGIONAL DEL BIO BIO |
| `teso_maule` | TESORERÍA REGIONAL DEL MAULE |
| `teso_ohiggins` | TESORERÍA REGIONAL LIB. GRAL BERNARDO OHIGGINS |
| `teso_santiago_oriente` | TESORERÍA REGIONAL SANTIAGO ORIENTE |
| `teso_santiago_poniente` | TESORERÍA REGIONAL SANTIAGO PONIENTE |
| `teso_santiago_sur` | TESORERÍA REGIONAL SANTIAGO SUR |

18 columnas binarias

In [ ]:
# TESORERIA_REGIONAL (18 categorías, en orden alfabético de LabelEncoder)
df = codificar_one_hot(df, 'TESORERIA_REGIONAL', [
    'teso_antofagasta',       # TESORERÍA REGIONAL DE ANTOFAGASTA
    'teso_arica_parinacota',  # ... DE ARICA Y PARINACOTA
    'teso_atacama',           # ... DE ATACAMA
    'teso_aysen',             # ... DE AYSEN
    'teso_coquimbo',          # ... DE COQUIMBO
    'teso_araucania',         # ... DE LA ARAUCANIA
    'teso_los_lagos',         # ... DE LOS LAGOS
    'teso_los_rios',          # ... DE LOS RIOS
    'teso_magallanes',        # ... DE MAGALLANES
    'teso_tarapaca',          # ... DE TARAPACA
    'teso_valparaiso',        # ... DE VALPARAÍSO
    'teso_nuble',             # ... DE ÑUBLE
    'teso_biobio',            # ... DEL BIO BIO
    'teso_maule',             # ... DEL MAULE
    'teso_ohiggins',          # ... LIB. GRAL BERNARDO OHIGGINS
    'teso_santiago_oriente',  # ... SANTIAGO ORIENTE
    'teso_santiago_poniente', # ... SANTIAGO PONIENTE
    'teso_santiago_sur',      # ... SANTIAGO SUR
])
df.head()

### b. `TIPO_PERSONA`
Categorías: `persona_juridica`, `persona_natural` → dos columnas binarias.

In [ ]:
df = codificar_one_hot(df, 'TIPO_PERSONA',
                       ['persona_juridica', 'persona_natural'])
df.head()